In [1]:

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, Pool

import subprocess
data_dir = "/l/izaac/data/24collision"
result = subprocess.run(['ls', data_dir], stdout=subprocess.PIPE, text=True)
file_list = [data_dir+"/"+file_path+":Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree" for file_path in result.stdout.split('\n') if file_path]

file_names = ["00222479_00000001_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1",
        "00222479_00000002_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1",
        "00222479_00000003_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1",
        "00222479_00000004_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1",
        "00222479_00000005_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;5",
        ]

file_names5 = ["00222479_00000001_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1",
        "00222479_00000002_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1",
        "00222479_00000003_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1",
        "00222479_00000004_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;1",
        "00222479_00000005_1.data_turbopass_bandq_lb2jpsilmd.root:Hlt2BandQ_Lb2JpsiLambdaTT/DecayTree;5"]

keys = ["Jpsi_PT","L_END_VRHO","L_BPVDIRA","L_BPVIP","L_BPVIPCHI2","L_MASS","Lb_BPVDIRA","Lb_BPVIP","Lb_BPVVDRHO","Lb_MAXDOCA","Lb_P","Lb_PT","Lb_CHI2","p_PID_P","p_PID_K","p_MINIP","p_P","p_PT","pim_P","pim_PT"]
total_keys = keys+['Lb_MASS']+['Jpsi_MASS']+['totCandidates']+['mup_P']+['mum_P']+['mum_PX']+['mum_PY']+['mum_PZ']+['mup_PX']+['mup_PY']+['mup_PZ']+['L_CHI2']+['Jpsi_CHI2']+['Jpsi_P']+['L_END_VZ']+['mup_PT']+['mum_PT']+['Jpsi_MAXDOCA']+['p_PX']+['p_PY']+['p_PZ']+['L_P']+['L_PX']+['L_PY']+['L_PZ']+['L_PT']+['Lb_PX']+['Lb_PY']+['Lb_PZ']+['mum_GHOSTPROB']+['L_BPVFDCHI2']+['Lb_BPVFDCHI2']+['Lb_BPVIPCHI2']+['mup_GHOSTPROB']+['p_GHOSTPROB']+['pim_GHOSTPROB']+['mum_PID_P']+['mum_PID_K']+['mum_PID_MU']+['mum_PID_E']+['mup_PID_P']+['mup_PID_K']+['mup_PID_MU']+['mup_PID_E']+['mup_CHI2']+['mup_MINIP']+['mup_MINIPCHI2']+['mum_CHI2']+['mum_MINIP']+['mum_MINIPCHI2']+['Jpsi_BPVDIRA']+['L_MINIPCHI2']+['Lb_MINIPCHI2']+['Jpsi_MINIPCHI2']+['nCandidate']+['EVENTNUMBER']+  ['Jpsi_END_VRHO'] + ['Jpsi_BPVIP']

model = CatBoostClassifier().load_model("TFG_model",format="cbm")
model.set_feature_names([name.replace('p_plus_','p_') for name in model.feature_names_])
model.set_feature_names([name.replace('pi_minus_','pim_') for name in model.feature_names_])
model.set_feature_names([name.replace('Lambda0_','L_') for name in model.feature_names_])
model.set_feature_names([name.replace('MAXSDOCA','MAXDOCA') for name in model.feature_names_])
# model.feature_names_

#pre_cut = f"(mup_PID_MU > 2) & (mum_PID_MU > 2) & (Jpsi_MAXDOCA < 0.15) & (Jpsi_MASS > 3050) & (Jpsi_MASS < 3150) & (p_PID_P > 2) & (L_PT > 1000) & (Lb_MINIPCHI2 < 1750 ) & (Lb_CHI2 < 150) & (Jpsi_PT > 1000) & (Jpsi_P > 18000) & (L_MASS < 1400)"

# #Izaac's cuts in cut varaible, Javier's cuts in pre_cut varaible, and mixing cuts in pre_cut variable 
# cut = f"(mup_PID_MU > 2) & (mum_PID_MU > 2) & (Jpsi_MAXDOCA < 0.15) & (Jpsi_MASS > 3050) & (Jpsi_MASS < 3150) & (p_PID_P > 2) & (L_PT > 1000) & (Lb_MINIPCHI2 < 1750 ) & (Lb_CHI2 < 150) & (bdt_score > 0.7) & (Jpsi_PT > 1000) & (Jpsi_P > 18000) & (nCandidate == 1) & (L_MASS < 1400)"
# pre_cut = f"(mup_PID_MU > 0) & (mum_PID_MU > 0) & (Jpsi_CHI2 < 4) & (Jpsi_BPVDIRA > 0.999) & (mup_P > 10000) & (mup_PT > 600) & (mum_P > 10000) & (mum_PT > 600) & (pim_P > 2000) & (pim_P < 500000) & (p_P > 10000)& (p_P < 500000) & (p_PT > 400) & (L_END_VZ > 5000) & (L_END_VZ < 8500) & (Cos_xi_L > 0.9999) & (Cos_xi_Lb > 0.99) & (L_PT > 450) & (Jpsi_MAXDOCA < 0.15) & (mum_GHOSTPROB < 0.4) & (L_MINIPCHI2 < 2*10e36) & (L_CHI2 < 75) & (Lb_MINIPCHI2 < 1750) & (Lb_CHI2 < 15) & (L_BPVIPCHI2 < 200) & (L_BPVIPCHI2 < 200) & (L_BPVFDCHI2 < 7500000000) & (Lb_BPVIPCHI2 < 1750) & (Lb_BPVFDCHI2 < 1500000000) "
pre_cut2 = "(Lb_MAXDOCA > 0.15) & (mup_PID_MU > 2) & (mum_PID_MU > 2) & (Jpsi_CHI2 < 4) & (mup_P > 10000) & (mup_PT > 600) & (mum_P > 10000) & (mum_PT > 600) & (pim_P > 2000) & (pim_P < 500000) & (p_P > 10000)& (p_P < 500000) & (p_PT > 400) & (L_END_VZ > 5500) & (L_END_VZ < 8500) & (Cos_xi_L > 0.9999) & (Cos_xi_Lb > 0.99) & (L_PT > 1000) & (Jpsi_MAXDOCA < 0.15) & (mum_GHOSTPROB < 0.4) & (pim_GHOSTPROB < 0.4) & (p_GHOSTPROB < 0.4) & (p_PID_P > 2)& (Lb_MINIPCHI2 < 1750 ) & (Lb_CHI2 < 15) & (Jpsi_PT > 1000) & (Jpsi_P > 18000) & (L_MASS < 1400)"
post_cut2 = f"(totCandidates == 1)"
# pre_cut3 = "(L_END_VRHO < 600) & (L_BPVIP < 30) & (L_BPVDIRA > 0.9999) & (Jpsi_END_VRHO > 0.6) & (Jpsi_BPVIP < 0.6) & (Jpsi_BPVDIRA > 0.999) & (Lb_MAXDOCA < 15) &  (mup_PID_MU > 2) & (mum_PID_MU > 2) & (Jpsi_CHI2 < 4) & (mup_P > 10000) & (mup_PT > 600) & (mum_P > 10000) & (mum_PT > 600) & (pim_P > 2000) & (pim_P < 500000) & (p_P > 10000)& (p_P < 500000) & (p_PT > 400) & (L_END_VZ > 5500) & (L_END_VZ < 8500) & (Cos_xi_L > 0.9999) & (Cos_xi_Lb > 0.99) & (L_PT > 1000) & (Jpsi_MAXDOCA < 0.15) & (mum_GHOSTPROB < 0.4)& (p_PID_P > 2)& (Lb_MINIPCHI2 < 15 ) & (Lb_CHI2 < 10) & (Jpsi_PT > 1000) & (Jpsi_P > 18000) & (L_MASS < 1400)"
pre_cut3 = "(Jpsi_PT > 1000) & (Jpsi_P > 18000) & (Jpsi_END_VRHO > 1) & (Jpsi_BPVIP < 0.6)& (Jpsi_BPVIP > 0.05) & (Jpsi_BPVDIRA > 0.999) & (Jpsi_MAXDOCA < 0.15)& (Jpsi_CHI2 < 4) & (L_END_VRHO > 150) &(L_END_VRHO < 600) & (L_BPVIP < 30) & (L_BPVDIRA > 0.99995) & (L_END_VZ > 5000) & (L_END_VZ < 8500) & (L_PT > 1500)& (L_P > 30000) & (L_MASS < 1200) & (Lb_MAXDOCA < 15) & (Lb_PT >2500) & (Lb_MINIPCHI2 < 6 )&(Lb_BPVVDRHO > 0.1 ) & (Lb_CHI2 < 15) &  (mup_PID_MU > 3) & (mum_PID_MU > 3)  & (mup_P > 10000) & (mup_PT > 600) & (mum_P > 10000) & (mum_PT > 600)&(mum_MINIP > 0.1)& (mup_MINIP > 0.1)& (mum_MINIPCHI2 > 100)& (mup_MINIPCHI2 > 100) &  (mum_GHOSTPROB < 0.4) & (pim_P > 2000) & (pim_P < 500000) & (p_P > 10000)& (p_P < 500000) & (p_PT > 400) & (p_PID_P > 2)"


# def appy_cut(x):
#     return x["(mup_PID_MU > 2) & (mum_PID_MU > 2) & (Jpsi_CHI2 < 4) & (mup_P > 10000) & (mup_PT > 600) & (mum_P > 10000) & (mum_PT > 600) & (pim_P > 2000) & (pim_P < 500000) & (p_P > 10000)& (p_P < 500000) & (p_PT > 400) & (L_END_VZ > 5500) & (L_END_VZ < 8500) & (Cos_xi_L > 0.9999) & (Cos_xi_Lb > 0.99) & (L_PT > 1000) & (Jpsi_MAXDOCA < 0.15) & (mum_GHOSTPROB < 0.4) & (p_PID_P > 2)& (Lb_MINIPCHI2 < 1750 ) & (Lb_CHI2 < 15) & (Jpsi_PT > 1000) & (Jpsi_P > 18000) & (L_MASS < 1400)"]

# def post_cut(x):
#     return x["(bdt_score > 0.9) & (totCandidates == 1)"]


bins = {
    "Lb_M" : np.linspace(5000,6500,25),
    "L_M" : np.linspace(850,1400,25),
    "Jpsi_M" : np.linspace(2990,3190,25)
}

histograms ={
    key : np.histogram(np.zeros(1000),bins[key])[0] for key in bins.keys()
}

bin_edges ={
    key : np.histogram(np.zeros(1000),bins[key])[1] for key in bins.keys()
}
# histograms_pre ={
#     key : np.histogram(np.zeros(1000),bins[key])[0] for key in bins.keys()
# }
bin_centres = {
   key : 0.5*(bins[key][1:] + bins[key][:-1]) for key in bins.keys()
}

for batch in uproot.iterate(files=file_names, expressions=total_keys +list(histograms.keys()), library="pd"): # apply preselection cuts here
     
    #Creates a new variables like the Cos()...
    sum_CHI2 = batch['Lb_CHI2']+batch['L_CHI2']+batch['Jpsi_CHI2']
    angle = np.arccos((batch['mum_PX']*batch['mup_PX']+batch['mum_PY']*batch['mup_PY']+batch['mum_PZ']*batch['mup_PZ'])/(batch['mum_P']*batch['mup_P']))
    cos_angle = ((batch['mum_PX']*batch['mup_PX']+batch['mum_PY']*batch['mup_PY']+batch['mum_PZ']*batch['mup_PZ'])/(batch['mum_P']*batch['mup_P']))
    cos_xi_L = ((batch['p_PX']*batch['L_PX']+batch['p_PY']*batch['L_PY']+batch['p_PZ']*batch['L_PZ'])/(batch['p_P']*batch['L_P']))
    cos_xi_Lb = ((batch['p_PX']*batch['Lb_PX']+batch['p_PY']*batch['Lb_PY']+batch['p_PZ']*batch['Lb_PZ'])/(batch['p_P']*batch['Lb_P']))

        #Add the new varaibles to signal_data
    batch['SUMCHI2'] = sum_CHI2
    batch['Theta_P'] = angle
    batch['Cos_theta_P'] = cos_angle
    batch['Cos_xi_L'] = cos_xi_L
    batch['Cos_xi_Lb'] = cos_xi_Lb

    batch_selected  = batch[batch.groupby(['EVENTNUMBER']).transform('min','SUMCHI2')['SUMCHI2']==batch['SUMCHI2']].drop_duplicates('EVENTNUMBER')  

    # preselected_batch = appy_cut(batch)
    preselected_batch = batch_selected.query(pre_cut3)

    All_predict_probs = model.predict_proba(preselected_batch)
    All_predict_probs_one = [1-i[0] for i in All_predict_probs]
    preselected_batch['bdt_score'] = All_predict_probs_one
    
    for key in histograms.keys():
        
        # hist, bin_edge = np.histogram(post_cut(preselected_batch)[key], bins=bins[key])
        hist, bin_edge = np.histogram(preselected_batch.query(post_cut2)[key], bins=bins[key]) 
        histograms[key]+= hist
        # batch.drop('bdt_score')

for key in histograms.keys():
    plt.errorbar(bin_centres[key], histograms[key], yerr=np.sqrt(histograms[key]),drawstyle = 'steps-mid',)
    #plt.errorbar(bin_centres[key], histograms_pre[key], yerr=np.sqrt(histograms_pre[key]),drawstyle = 'steps-mid',)
    plt.ylim([0,np.max(histograms[key])*1.1])
    plt.xlabel(key)
    plt.ylabel('Entries')
    plt.title(f'Histogram of {key}')
    plt.show()


NameError: name 'data24' is not defined